# Fase 6: Métricas Específicas - Centralidad y Corredores Críticos
**Tema**: 2. Centralidad y corredores críticos de la red vial
**Requisito de Rúbrica**: Métrica dependiente del tema con parámetros justificados.

Calcularemos las métricas de centralidad que permitirán identificar los corredores críticos:
1. **Betweenness Centrality (Intermediación)**
2. **Closeness Centrality (Cercanía)**

**Justificación del Rigor Algorítmico**: El cálculo exacto tiene complejidad $O(NM)$. Dado que nuestro grafo tiene $N \approx 2500$ nodos, calcularemos la medida exacta utilizando la longitud (`length`), evitando la aproximación por muestreo ($k$).

In [1]:
import osmnx as ox
import networkx as nx
import pandas as pd
import time

G = ox.load_graphml("../data/miraflores_drive_poi.graphml")
print(f"Grafo cargado: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas.")

Grafo cargado: 1369 nodos, 2678 aristas.


## Betweenness Centrality (Intermediación)

In [2]:
print("Calculando Betweenness Centrality exacto...")
start_time = time.time()
bc = nx.betweenness_centrality(G, weight='length', normalized=True)
nx.set_node_attributes(G, bc, 'betweenness')
print(f"Completado en {time.time() - start_time:.2f} segundos.")

Calculando Betweenness Centrality exacto...


Completado en 9.70 segundos.


## Closeness Centrality (Cercanía)

In [3]:
print("Calculando Closeness Centrality exacto...")
start_time = time.time()
cc = nx.closeness_centrality(G, distance='length')
nx.set_node_attributes(G, cc, 'closeness')
print(f"Completado en {time.time() - start_time:.2f} segundos.")

Calculando Closeness Centrality exacto...


Completado en 3.47 segundos.


## Identificación de Corredores Críticos

In [4]:
nodes, edges = ox.graph_to_gdfs(G)
threshold = nodes['betweenness'].quantile(0.95)
print(f"Umbral de Betweenness (Top 5%): {threshold:.4f}")

critical_nodes = {n: (1 if bc_val >= threshold else 0) for n, bc_val in bc.items()}
nx.set_node_attributes(G, critical_nodes, 'is_critical')

n_critical = sum(critical_nodes.values())
print(f"Se identificaron {n_critical} nodos críticos.")

Umbral de Betweenness (Top 5%): 0.0741
Se identificaron 69 nodos críticos.


## Guardar Grafo Analizado

In [5]:
out_path = "../data/miraflores_drive_centralities.graphml"
ox.save_graphml(G, out_path)
print(f"Grafo guardado en {out_path}")

Grafo guardado en ../data/miraflores_drive_centralities.graphml
